# GRI → ESRS Mapping Notebook
### Master's Thesis — Multi-Company

**Companies covered:** Nomad Foods (2024), Nomad Foods (2025), Profand (2024), Profand (2025), Thai Union

This notebook:
1. Loads `GRI_ESRS_v6.xlsx` (`*_GRI_Index` sheets + extended `Interop_Index`)
2. For **each company**: merges its GRI Index with the Interoperability Index
3. Extracts only topical ESRS standards (E1–E5, S1–S4, G1) and **explodes** multi-DR rows
4. Exports one Excel with **one tab per company** + a **cross-company comparison tab**


**Note (added on verification pass, 2026-07-30):** the bridge workbook has been renamed GRI_ESRS_v5.xlsx -> v6 since this notebook was last edited -- update the upload step to v6. Also: the `OUTPUT_FILE` this notebook writes (`GRI_ESRS_Topical_Mapping_AllCompanies.xlsx`) is no longer a separate file in practice -- its output sheets were merged directly into the v6 bridge workbook itself (the `_Topical_Mapping` sheets there). If you re-run this notebook, either keep that manual merge step or update this notebook to write into the bridge workbook directly.

## Cell 1 — Install and import packages

In [1]:
!pip install openpyxl xlsxwriter --quiet

import pandas as pd
import re
import os
from google.colab import files
from IPython.display import display

print('All packages ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.9 MB/s eta 0:00:00
✅ All packages ready.


## Cell 2 — Upload your Excel file

Upload: **`GRI_ESRS_v6.xlsx`**

In [2]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'Uploaded: {filename}')

Saving GRI_ESRS_v5.xlsx to GRI_ESRS_v5.xlsx
Uploaded: GRI_ESRS_v5.xlsx


## Cell 3 — Define companies to process

Add or remove companies here. Each entry is `(company_name, sheet_name_in_Excel)`.

In [3]:
# ── Define which companies to process ────────────────────────────────────────
# Format: (display_name, sheet_name_in_Excel)
COMPANIES = [
    ('Nomad Foods',       'Nomad_GRI_Index'),        # 2024 reporting
    ('Nomad Foods 2025',  'Nomad_GRI_Index_2025'),   # 2025 reporting (NEW)
    ('Profand',           'Profand_GRI_Index'),
    ('Profand 2025',      'Profand_GRI_Index_2025'),
    ('Thai Union',        'ThaiUnion_GRI_Index'),    # NEW (FY2024, reported-only)
]

print(f'Companies to process: {[c[0] for c in COMPANIES]}')


Companies to process: ['Nomad Foods', 'Nomad Foods 2025', 'Profand', 'Thai Union']


## Cell 4 — Load Interoperability Index (shared lookup)

In [4]:
# Load the shared GRI → ESRS crosswalk (same for all companies)
interop = pd.read_excel(filename, sheet_name='Interop_Index')
interop = interop.fillna('—')

print(f'Interop Index rows: {len(interop)}')
print(f'Columns: {list(interop.columns)}')
display(interop.head(3))

Interop Index rows: 144
Columns: ['Lookup_Key', 'ESRS Topic', 'DR', 'ESRS Paragraph Reference', 'Notes Code', 'Notes Explanation']


,Lookup_Key,ESRS Topic,DR,ESRS Paragraph Reference,Notes Code,Notes Explanation
0,GRI2|2-1,—,—,See requirements of Directive 2013/34/EU,—,—
1,GRI2|2-2,ESRS 1 / ESRS 2,BP-1,ESRS 1 5.1; ESRS 2 BP-1 §5 (a) and (b) i,—,—
2,GRI2|2-3,ESRS 1,—,ESRS 1 §73,—,—


## Cell 5 — Core functions

In [6]:
def extract_topical_drs(dr_string):
    """From 'SBM-1 / E1-4 / E1-6 / MDR-P' keep only topical parts 'E1-4 / E1-6'."""
    if not dr_string or str(dr_string).strip() in ('—', 'nan', ''):
        return '—'
    dr_clean = re.sub(r'\(.*?\)', '', str(dr_string))
    parts = [p.strip() for p in dr_clean.split('/')]
    topical = [p for p in parts if re.match(r'^[ESG]\d-\d', p.strip())]
    return ' / '.join(topical) if topical else '—'


def extract_standard(topical_dr):
    """From 'E1-4 / E1-6 / S1-3' -> 'E1 / S1' (unique top-level standards)."""
    if topical_dr == '—':
        return '—'
    parts = topical_dr.split(' / ')
    standards = list(dict.fromkeys([p[:2] for p in parts if len(p) >= 2]))
    return ' / '.join(standards)


def process_company(company_name, sheet_name, interop_df, filename):
    """Load one company GRI index, merge with interop, explode to one row per topical DR.

    v5 change: rows are EXPLODED so each topical DR (e.g. E1-4, E1-6) gets its own
    row — matching the layout of the *_Topical_Mapping and Mapping_* sheets.
    """
    print(f'  Loading {company_name} from sheet: {sheet_name}')
    gri = pd.read_excel(filename, sheet_name=sheet_name).fillna('')
    merged = gri.merge(interop_df, on='Lookup_Key', how='left')
    for col in ['ESRS Topic','DR','ESRS Paragraph Reference','Notes Code','Notes Explanation']:
        merged[col] = merged[col].fillna('—')

    merged['Topical_DR']    = merged['DR'].apply(extract_topical_drs)
    merged['Is_Topical']    = merged['Topical_DR'] != '—'
    merged['ESRS_Standard'] = merged['Topical_DR'].apply(extract_standard)
    merged['Company']       = company_name

    topical = merged[merged['Is_Topical']].copy()
    rows = []
    for _, r in topical.iterrows():
        for dr in r['Topical_DR'].split(' / '):
            rr = r.copy()
            rr['DR_single'] = dr
            rr['ESRS_Standard_single'] = dr[:2]
            rows.append(rr)
    topical_x = pd.DataFrame(rows).reset_index(drop=True) if rows else pd.DataFrame()
    if len(topical_x):
        for m in ['Page_Ref','Coverage','Comments']:
            topical_x[m] = ''

    print(f'    GRI rows: {len(merged)} | topical(pre-explode): {len(topical)} | exploded: {len(topical_x)}')
    return merged, topical_x


print('✅ Functions defined (v5: with DR explode).')


✅ Functions defined (v5: with DR explode).


## Cell 6 — Process all companies

In [7]:
all_full    = {}   # company_name -> full merged df (all rows)
all_topical = {}   # company_name -> topical-only df

print('Processing companies...')
print()

for company_name, sheet_name in COMPANIES:
    print(f'── {company_name} ──')
    full_df, topical_df = process_company(company_name, sheet_name, interop, filename)
    all_full[company_name]    = full_df
    all_topical[company_name] = topical_df
    print()

print('✅ All companies processed.')

Processing companies...

── Nomad Foods ──
  Loading Nomad Foods from sheet: Nomad_GRI_Index
    GRI rows: 85 | topical(pre-explode): 54 | exploded: 98

── Nomad Foods 2025 ──
  Loading Nomad Foods 2025 from sheet: Nomad_GRI_Index_2025
    GRI rows: 86 | topical(pre-explode): 51 | exploded: 96

── Profand ──
  Loading Profand from sheet: Profand_GRI_Index
    GRI rows: 75 | topical(pre-explode): 48 | exploded: 76

── Thai Union ──
  Loading Thai Union from sheet: ThaiUnion_GRI_Index
    GRI rows: 126 | topical(pre-explode): 85 | exploded: 158

✅ All companies processed.


## Cell 7 — Preview results per company

In [8]:
PREVIEW_COLS = ['GRI Standard','GRI No.','Description','ESRS Topic','ESRS_Standard_single','DR_single']

for company_name, _ in COMPANIES:
    df = all_topical[company_name]
    print(f'\n{"="*60}')
    print(f' {company_name} — topical rows ({len(df)})')
    print(f'{"="*60}')
    cols = [c for c in PREVIEW_COLS if c in df.columns]
    display(df[cols].head(10))



 Nomad Foods — topical rows (98)


,GRI Standard,GRI No.,Description,ESRS Topic,ESRS_Standard_single,DR_single
0,GRI 2: General Disclosures 2021,2-7,Total number of employees,ESRS 2 / ESRS S1,S1,S1-6
1,GRI 2: General Disclosures 2021,2-7,Employee breakdown by gender and region,ESRS 2 / ESRS S1,S1,S1-6
2,GRI 2: General Disclosures 2021,2-13,Delegation of responsibility,ESRS 2 / ESRS G1,G1,G1-3
3,GRI 2: General Disclosures 2021,2-23,Policy commitments for responsible business co...,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,S1,S1-1
4,GRI 2: General Disclosures 2021,2-23,Policy commitments for responsible business co...,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,S2,S2-1
5,GRI 2: General Disclosures 2021,2-23,Policy commitments for responsible business co...,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1
6,GRI 2: General Disclosures 2021,2-24,Embedding policy commitments for responsible b...,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1
7,GRI 2: General Disclosures 2021,2-26,Mechanisms for seeking advice and raising conc...,ESRS S1 / S2 / S3 / S4 / G1,S1,S1-3
8,GRI 2: General Disclosures 2021,2-26,Mechanisms for seeking advice and raising conc...,ESRS S1 / S2 / S3 / S4 / G1,S2,S2-3
9,GRI 2: General Disclosures 2021,2-26,Mechanisms for seeking advice and raising conc...,ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1



 Nomad Foods 2025 — topical rows (96)


,GRI Standard,GRI No.,Description,ESRS Topic,ESRS_Standard_single,DR_single
0,GRI 2: General Disclosures 2021,2-7,Total number of employees,ESRS 2 / ESRS S1,S1,S1-6
1,GRI 2: General Disclosures 2021,2-7,Employee breakdown by gender and region,ESRS 2 / ESRS S1,S1,S1-6
2,GRI 2: General Disclosures 2021,2-13,Delegation of responsibility,ESRS 2 / ESRS G1,G1,G1-3
3,GRI 2: General Disclosures 2021,2-23,Policy commitments for responsible business co...,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,S1,S1-1
4,GRI 2: General Disclosures 2021,2-23,Policy commitments for responsible business co...,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,S2,S2-1
5,GRI 2: General Disclosures 2021,2-23,Policy commitments for responsible business co...,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1
6,GRI 2: General Disclosures 2021,2-24,Embedding policy commitments for responsible b...,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1
7,GRI 2: General Disclosures 2021,2-26,Mechanisms for seeking advice and raising conc...,ESRS S1 / S2 / S3 / S4 / G1,S1,S1-3
8,GRI 2: General Disclosures 2021,2-26,Mechanisms for seeking advice and raising conc...,ESRS S1 / S2 / S3 / S4 / G1,S2,S2-3
9,GRI 2: General Disclosures 2021,2-26,Mechanisms for seeking advice and raising conc...,ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1



 Profand — topical rows (76)


,GRI Standard,GRI No.,Description,ESRS Topic,ESRS_Standard_single,DR_single
0,GRI 2: General Disclosures 2021,2-7,Employees,ESRS 2 / ESRS S1,S1,S1-6
1,GRI 2: General Disclosures 2021,2-8,Non-employee workers,ESRS S1,S1,S1-7
2,GRI 2: General Disclosures 2021,2-13,Delegation of responsibility for impact manage...,ESRS 2 / ESRS G1,G1,G1-3
3,GRI 2: General Disclosures 2021,2-21,Total annual compensation ratio,ESRS S1,S1,S1-16
4,GRI 2: General Disclosures 2021,2-23,Commitments and policies,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,S1,S1-1
5,GRI 2: General Disclosures 2021,2-23,Commitments and policies,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,S2,S2-1
6,GRI 2: General Disclosures 2021,2-23,Commitments and policies,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1
7,GRI 2: General Disclosures 2021,2-24,Incorporating commitments and policies,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1
8,GRI 2: General Disclosures 2021,2-25,Processes to remedy negative impacts,ESRS S1 / S2 / S3 / S4,S1,S1-1
9,GRI 2: General Disclosures 2021,2-25,Processes to remedy negative impacts,ESRS S1 / S2 / S3 / S4,S1,S1-3



 Thai Union — topical rows (158)


,GRI Standard,GRI No.,Description,ESRS Topic,ESRS_Standard_single,DR_single
0,GRI 2: General Disclosures 2021,2-7,Employees,ESRS 2 / ESRS S1,S1,S1-6
1,GRI 2: General Disclosures 2021,2-8,Workers who are not employees,ESRS S1,S1,S1-7
2,GRI 2: General Disclosures 2021,2-13,Delegation of responsibility for managing impacts,ESRS 2 / ESRS G1,G1,G1-3
3,GRI 2: General Disclosures 2021,2-16,Communication of critical concerns,ESRS 2 / ESRS G1,G1,G1-1
4,GRI 2: General Disclosures 2021,2-16,Communication of critical concerns,ESRS 2 / ESRS G1,G1,G1-3
5,GRI 2: General Disclosures 2021,2-21,Annual total compensation ratio,ESRS S1,S1,S1-16
6,GRI 2: General Disclosures 2021,2-23,Policy commitments,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,S1,S1-1
7,GRI 2: General Disclosures 2021,2-23,Policy commitments,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,S2,S2-1
8,GRI 2: General Disclosures 2021,2-23,Policy commitments,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1
9,GRI 2: General Disclosures 2021,2-24,Embedding policy commitments,ESRS 2 / ESRS S1 / S2 / S3 / S4 / G1,G1,G1-1


## Cell 8 — Cross-company summary

In [9]:
print('='*60)
print('CROSS-COMPANY SUMMARY')
print('='*60)

ESRS_LABELS = {
    'E1': 'E1  Climate change',
    'E2': 'E2  Pollution',
    'E3': 'E3  Water & marine resources',
    'E4': 'E4  Biodiversity & ecosystems',
    'E5': 'E5  Resource use & circular economy',
    'S1': 'S1  Own workforce',
    'S2': 'S2  Value chain workers',
    'S3': 'S3  Affected communities',
    'S4': 'S4  Consumers & end-users',
    'G1': 'G1  Business conduct',
}

# Build comparison table
summary_data = {}
for company_name, _ in COMPANIES:
    df = all_topical[company_name]
    col = 'ESRS_Standard_single' if 'ESRS_Standard_single' in df.columns else 'ESRS_Standard'
    counts = df[col].value_counts()
    summary_data[company_name] = counts

summary_df = pd.DataFrame(summary_data).fillna(0).astype(int)
summary_df.index = [ESRS_LABELS.get(i, i) for i in summary_df.index]
summary_df = summary_df.sort_index()
summary_df.index.name = 'ESRS Standard'

print()
display(summary_df)
print()
print('Total topical rows per company:')
for company_name, _ in COMPANIES:
    print(f'  {company_name}: {len(all_topical[company_name])} rows')

CROSS-COMPANY SUMMARY



,Nomad Foods,Nomad Foods 2025,Profand,Thai Union
ESRS Standard,,,,
E1 Climate change,19,17,8,20
E2 Pollution,2,2,2,3
E3 Water & marine resources,7,7,3,8
E4 Biodiversity & ecosystems,6,2,1,9
E5 Resource use & circular economy,13,13,4,14
G1 Business conduct,11,12,11,20
S1 Own workforce,26,31,28,55
S2 Value chain workers,8,8,8,10
S3 Affected communities,0,0,6,12



Total topical rows per company:
  Nomad Foods: 98 rows
  Nomad Foods 2025: 96 rows
  Profand: 76 rows
  Thai Union: 158 rows


## Cell 9 — Export to Excel (one tab per company + cross-company tab)

In [10]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.formatting.rule import CellIsRule

OUTPUT_FILE = 'GRI_ESRS_Topical_Mapping_AllCompanies.xlsx'

# ── Styles ────────────────────────────────────────────────────────────────────
BD_S = Side(style='thin', color='C5D9ED')
BD   = Border(left=BD_S, right=BD_S, top=BD_S, bottom=BD_S)
WRAP = Alignment(wrap_text=True, vertical='top')
CWRAP= Alignment(wrap_text=True, vertical='top', horizontal='center')
BLUE = PatternFill('solid', fgColor='2E5F8A')
GREEN= PatternFill('solid', fgColor='1A5C3A')
YELL = PatternFill('solid', fgColor='7B5800')
WF   = Font(bold=True, color='FFFFFF', name='Arial', size=10)
EVEN = PatternFill('solid', fgColor='F4F8FC')
ODD  = PatternFill('solid', fgColor='FFFFFF')
DF   = Font(name='Arial', size=9)
MF   = Font(name='Arial', size=9, color='7B4A00')

EXPORT_COLS = [
    ('Category',                  18,  BLUE),
    ('GRI Standard',              35,  BLUE),
    ('GRI No.',                   14,  BLUE),
    ('Description',               45,  BLUE),
    ('Location in Report',        35,  BLUE),
    ('ESRS Topic',                30,  GREEN),
    ('ESRS_Standard',             14,  GREEN),
    ('Topical_DR',                28,  GREEN),
    ('ESRS Paragraph Reference',  65,  GREEN),
    ('Notes Code',                12,  GREEN),
    ('Notes Explanation',         50,  GREEN),
    ('Page_Ref',                  10,  YELL),
    ('Coverage',                  12,  YELL),
    ('Comments',                  25,  YELL),
]

def write_company_sheet(wb, sheet_name, df):
    ws = wb.create_sheet(sheet_name)

    # Headers
    for col_idx, (col_name, width, hfill) in enumerate(EXPORT_COLS, 1):
        c = ws.cell(row=1, column=col_idx, value=col_name)
        c.font = WF; c.fill = hfill; c.alignment = CWRAP; c.border = BD
        ws.column_dimensions[get_column_letter(col_idx)].width = width
    ws.row_dimensions[1].height = 30

    # Data rows
    manual = {'Page_Ref', 'Coverage', 'Comments'}
    col_names = [c[0] for c in EXPORT_COLS]

    for row_idx, (_, row) in enumerate(df.iterrows(), 2):
        fill = EVEN if row_idx % 2 == 0 else ODD
        for col_idx, col_name in enumerate(col_names, 1):
            val = row.get(col_name, '')
            val = '' if pd.isna(val) else val
            c = ws.cell(row=row_idx, column=col_idx, value=val)
            c.fill = fill; c.alignment = WRAP; c.border = BD
            c.font = MF if col_name in manual else DF
        ws.row_dimensions[row_idx].height = 38

    # Dropdown for Coverage
    cov_idx = col_names.index('Coverage') + 1
    cov_col = get_column_letter(cov_idx)
    dv = DataValidation(type='list', formula1='"Full,Partial,Gap,N/A"', allow_blank=True)
    dv.sqref = f'{cov_col}2:{cov_col}{len(df)+1}'
    ws.add_data_validation(dv)

    # Conditional formatting
    cf = f'{cov_col}2:{cov_col}{len(df)+1}'
    ws.conditional_formatting.add(cf, CellIsRule('equal', ['"Full"'],    fill=PatternFill('solid',fgColor='D5F5E3'), font=Font(color='1E8449',name='Arial',size=9,bold=True)))
    ws.conditional_formatting.add(cf, CellIsRule('equal', ['"Partial"'], fill=PatternFill('solid',fgColor='FEF9E7'), font=Font(color='B7950B',name='Arial',size=9,bold=True)))
    ws.conditional_formatting.add(cf, CellIsRule('equal', ['"Gap"'],     fill=PatternFill('solid',fgColor='FDDCDC'), font=Font(color='C0392B',name='Arial',size=9,bold=True)))

    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    return ws


# ── Build workbook ────────────────────────────────────────────────────────────
wb = openpyxl.Workbook()
wb.remove(wb.active)  # remove default sheet

# One sheet per company
for company_name, _ in COMPANIES:
    df = all_topical[company_name]
    sheet_name = company_name.replace(' ', '_')[:31]
    write_company_sheet(wb, sheet_name, df)
    print(f'  ✅ Sheet written: {sheet_name} ({len(df)} rows)')

# Cross-company stacked sheet
all_stacked = pd.concat(
    [all_topical[c].assign(Company=c) for c, _ in COMPANIES],
    ignore_index=True
)
write_company_sheet(wb, 'ALL_Companies', all_stacked)
print(f'  ✅ Sheet written: ALL_Companies ({len(all_stacked)} rows)')

wb.save(OUTPUT_FILE)
print(f'\n✅ Excel saved: {OUTPUT_FILE}')

  ✅ Sheet written: Nomad_Foods (98 rows)
  ✅ Sheet written: Nomad_Foods_2025 (96 rows)
  ✅ Sheet written: Profand (76 rows)
  ✅ Sheet written: Thai_Union (158 rows)
  ✅ Sheet written: ALL_Companies (428 rows)

✅ Excel saved: GRI_ESRS_Topical_Mapping_AllCompanies.xlsx


## Cell 10 — Download the output Excel

In [11]:
files.download(OUTPUT_FILE)
print('✅ Download started.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started.


## Cell 11 — Summary statistics for your thesis

In [12]:
print('='*65)
print('THESIS SUMMARY — GRI → ESRS Topical Mapping')
print('='*65)

for company_name, _ in COMPANIES:
    full = all_full[company_name]
    topical = all_topical[company_name]

    print(f'\n{company_name}')
    print(f'  Total GRI disclosures reported:  {len(full)}')
    print(f'  Map to topical ESRS standard:    {len(topical)}')
    print(f'  No topical ESRS coverage:        {len(full) - len(topical)}')

    # Notes code breakdown
    notes = topical['Notes Code'].value_counts()
    if len(notes):
        print(f'  GRI vs ESRS differences:')
        notes_map = {'1a':'more granularity required','1b':'GRI quant vs ESRS qualitative',
                     '2a':'broader scope','2b':'same objective diff formulation','2c':'worker coverage diff'}
        for code, cnt in notes.items():
            if str(code) not in ('—','nan',''):
                print(f'    {code} ({notes_map.get(str(code).strip(), code)}): {cnt} rows')

print()
print('Cross-company coverage matrix:')
display(summary_df)

THESIS SUMMARY — GRI → ESRS Topical Mapping

Nomad Foods
  Total GRI disclosures reported:  85
  Map to topical ESRS standard:    98
  No topical ESRS coverage:        -13
  GRI vs ESRS differences:
    1a (more granularity required): 9 rows
    2b (same objective diff formulation): 4 rows
    1a / 2a (1a / 2a): 4 rows
    2a (broader scope): 4 rows
    1b (GRI quant vs ESRS qualitative): 3 rows
    1a / 2c (1a / 2c): 2 rows
    1a / 1b (1a / 1b): 1 rows

Nomad Foods 2025
  Total GRI disclosures reported:  86
  Map to topical ESRS standard:    96
  No topical ESRS coverage:        -10
  GRI vs ESRS differences:
    1a (more granularity required): 9 rows
    1a / 2a (1a / 2a): 4 rows
    1b (GRI quant vs ESRS qualitative): 4 rows
    2a (broader scope): 4 rows
    2b (same objective diff formulation): 3 rows
    1a / 2c (1a / 2c): 2 rows
    1a / 1b (1a / 1b): 1 rows

Profand
  Total GRI disclosures reported:  75
  Map to topical ESRS standard:    76
  No topical ESRS coverage:        -

,Nomad Foods,Nomad Foods 2025,Profand,Thai Union
ESRS Standard,,,,
E1 Climate change,19,17,8,20
E2 Pollution,2,2,2,3
E3 Water & marine resources,7,7,3,8
E4 Biodiversity & ecosystems,6,2,1,9
E5 Resource use & circular economy,13,13,4,14
G1 Business conduct,11,12,11,20
S1 Own workforce,26,31,28,55
S2 Value chain workers,8,8,8,10
S3 Affected communities,0,0,6,12
